# Bag Of Words

Bag of Words (BoW) is a simple text representation that converts each document into a vector of word counts (or presence/absence), ignoring word order and grammar. It treats a document as an unordered “bag” of tokens and uses the frequency of each token as its feature.

## How Bag of Words works
Given a corpus (collection of documents):

- Tokenize each document into words.

- Build a vocabulary: list of all unique words across the corpus (often sorted).

- For each document, create a vector of length |vocabulary|, where each entry is:

    - Count of that word in the document, or

    - Binary (1 if present, 0 if not).

### Example with two sentences:

Doc1: “the cat sat”

Doc2: “the dog sat”

Vocabulary (sorted): ["cat", "dog", "sat", "the"]

**BoW** vectors (counts):

Doc1 → [1, 0, 1, 1]

Doc2 → [0, 1, 1, 1]

- Order in the sentence is lost; only counts matter.

## Why it’s used
- Converts text into a fixed-length numeric vector that ML algorithms can consume.

- Works as a baseline for tasks like:

    - Text classification (spam detection, topic labeling)

    - Sentiment analysis

    - Document similarity and clustering

    - Information retrieval

In scikit-learn, this is implemented via CountVectorizer, which builds the vocabulary and returns BoW matrices.

## Advantages of Bag of Words
- Simple and intuitive

    - Easy to understand: just count words.

    - Straightforward to implement and debug.

- Fast and efficient for many tasks

    - Minimal preprocessing beyond tokenization and basic cleaning.

    - Computationally efficient, especially with sparse matrix implementations.

- Versatile baseline

    - Applicable to many NLP tasks (classification, retrieval, clustering).

    - Often used as a first model before moving to more complex representations (TF–IDF, embeddings).

- Interpretable features

    - Each dimension corresponds to a specific word; you can inspect which words drive predictions.

## Disadvantages of Bag of Words
- Ignores word order and context

    - “dog bites man” and “man bites dog” get the same representation.

    - Cannot capture syntax, phrases, or semantic relationships.

- High dimensionality

    - Vector length = vocabulary size, which can be tens or hundreds of thousands.

    - Leads to large, sparse matrices and can hurt generalization on small datasets.

- Sparse representations

    - Most entries are zero for any given document.

    - Some models handle sparsity well; others don’t, and storage/computation can still be costly.

- No semantic understanding

    - Synonyms are treated as unrelated (“car” vs “automobile”).

    - Polysemy (same word, different meanings) is ignored.

- Sensitive to vocabulary design

    - Including rare words increases sparsity and noise; excluding important words loses signal.

    - Requires choices about lowercasing, stop words, min/max frequency, etc.

## Typical use pattern in practice
- Use BoW (or TF–IDF, which builds on BoW) as a baseline model for text classification or similarity.

    - If performance is insufficient or semantics matter a lot, move to:

    - N-grams (to capture some local order)

    - Word embeddings (Word2Vec, GloVe, FastText)

    - Contextual models (BERT, etc.)

In [ ]:
!pip3 install pandas
import pandas

In [3]:
messages = pandas.read_csv('data/sms_spam_collection/sms_collection', sep='\t',names=['label', 'message'])

In [4]:
messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [11]:
## data cleaning and preprocessing
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

In [7]:
lemmatizer = WordNetLemmatizer()

In [9]:
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [12]:
# go thru each message -> remove non-alphabetic characters -> lowercase -> tokenize -> remove stopwords -> lemmatize -> join back to string
corpus = []
for i in range(len(messages)):
    message = re.sub('[^a-zA-Z]', ' ', messages['message'][i])
    message = message.lower()
    message = nltk.word_tokenize(message)
    tagged = nltk.pos_tag(message)
    message = [lemmatizer.lemmatize(word, pos=get_wordnet_pos(tag)) for word, tag in tagged if word not in set(stopwords.words('english'))]
    corpus.append(' '.join(message))

In [13]:
corpus[:10]

['go jurong point crazy available bugis n great world la e buffet cine get amore wat',
 'ok lar joking wif u oni',
 'free entry wkly comp win fa cup final tkts st may text fa receive entry question std txt rate c apply',
 'u dun say early hor u c already say',
 'nah think go usf live around though',
 'freemsg hey darling week word back like fun still tb ok xxx std chgs send rcv',
 'even brother like speak treat like aid patent',
 'per request melle melle oru minnaminunginte nurungu vettam set callertune caller press copy friend callertune',
 'winner value network customer select receivea prize reward claim call claim code kl valid hour',
 'mobile month u r entitle update late colour mobile camera free call mobile update co free']

## Apply BoW
In scikit-learn, “BOW” (bag-of-words) is implemented mainly via CountVectorizer in sklearn.feature_extraction.text. It converts a collection of text documents into a sparse matrix where each column is a token (word or n‑gram) and each row is a document, with cell values equal to token counts.

### Common options you’ll care about
- ngram_range=(1, 2) – use unigrams + bigrams (or (2,2) for only bigrams). This one is **Important**, we will see this next.

- stop_words="english" – drop common low‑info words like “the”, “is”.

- max_features=5000 – cap vocabulary size to control dimensionality.

- min_df, max_df – ignore tokens that are too rare or too common across documents.

- binary=True – store 0/1 instead of counts (presence/absence BOW).

In [16]:
!pip3 install scikit-learn
from sklearn.feature_extraction.text import CountVectorizer

In [18]:
count_vectorizer = CountVectorizer(max_features=2500)
# here max_features=2500 means we will only keep the 2500 most frequent words in our vocabulary. 
# This is a common practice to reduce the dimensionality of the feature space and to focus on the most informative words.

In [19]:
X_bow = count_vectorizer.fit_transform(corpus).toarray()

In [20]:
X_bow.shape

(5572, 2500)

- The shape is (no. of sentences * vocab size)

- The vocabulary size is 2500 as we intended

In [25]:
not_zero = [X_bow[2][i] for i in range(len(X_bow[2])) if X_bow[2][i] != 0]
not_zero

[np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(2),
 np.int64(2),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1)]

We can see there are counts other than 1

In [22]:
# lets apply the Binary BoW using the same CountVectorizer but with binary=True parameter. This will create a binary representation of the BoW, where each feature indicates the presence (1) or absence (0) of a word in the document.
count_vectorizer_binary = CountVectorizer(max_features=2500, binary=True)
X_bow_binary = count_vectorizer_binary.fit_transform(corpus).toarray()

In [26]:
not_zero_binary = [X_bow_binary[2][i] for i in range(len(X_bow_binary[2])) if X_bow_binary[2][i] != 0]
not_zero_binary

[np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1)]